# Retrieval Strategies Benchmark

## Purpose
Measure whether rewriting the user's question with an LLM before vector search retrieves the passages needed to answer it more reliably than searching with the raw question.

## Test Setup

**Document:** *Adversarial Robustness of AI-Generated Image Detectors in the Real World* (`2410.01574v4.pdf`), the same paper used in the text splitting benchmark.

**Fixed across all strategies** (only the query side changes):

| Component | Setting |
|---|---|
| Conversion | `PDF2MarkdownProcessor` |
| Chunking | `split_markdown_by_headings_with_token_limit`, 512 tokens (Gemma 3 tokenizer) -> 46 chunks |
| Embeddings | `qwen3-embedding:0.6b` via Ollama (1024 dimensions), no query instruction prefix |
| Vector store | ChromaDB in-memory collection, default L2 distance |
| Query LLM | `gemma3:4b` via Ollama, `temperature=0.3`, `seed=42`, JSON-schema-constrained output |
| Test questions | 18 hand-written questions with gold chunk labels (Section 4) |

**Strategies compared:**

1. **Baseline**: embed the raw question, take the top 5 chunks.
2. **Rewriting**: the LLM rewrites the question into one search query, then top 5.
3. **Multi-query**: the LLM generates 3-5 queries from different angles (decomposition, terminology, step-back, answer-style). Each query plus the original retrieves its top 10, and the lists are merged with Reciprocal Rank Fusion (RRF) into a top 5.

**Metrics** (k = 5): Hit@1, Hit@5, Recall@5 (share of gold chunks retrieved), MRR, and query-side latency.

## 1. Index the Corpus

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.PDF2MarkdownProcessor import PDF2MarkdownProcessor
from academic_research_assistant import embed_documents, markdown_splitter
from dotenv import load_dotenv
import os

c:\Users\youss\OneDrive\Desktop\Academic_Knowledge_&_Research_Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set up ChromaDB client and create a collection
import chromadb
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="documents")

In [3]:
# Set up the tokenizer and token limit
from transformers import AutoTokenizer

load_dotenv()

HF_TOKEN = os.getenv("HG_ACCESS_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-4b-it",
    token=HF_TOKEN
)

token_limit = 512

In [4]:
# Embed the documents and add them to the ChromaDB collection
pdf_path = Path.cwd().parent / "documents" / "pdfs" / "2410.01574v4.pdf"
converter = PDF2MarkdownProcessor()
embedded_document = embed_documents.embed_document(pdf_path, converter, tokenizer, token_limit)
print(f"Total chunks created: {len(embedded_document)}")

Total chunks created: 46


In [5]:
# Print the embedded document chunks for verification
for chunk in embedded_document:
    print(f"Chunk index: {chunk.chunk_index}")
    print(f"Source: {chunk.source}")
    print(f"Text snippet: {chunk.text[:50]}...")  
    print(f"Embedding vector length: {len(chunk.embedding)}")
    print(f"Embedding snippet: {chunk.embedding[:5]}...")
    print(f"Token count: {chunk.token_count}") 
    print("-" * 40)  

Chunk index: 0
Source: 2410.01574v4.pdf
Text snippet: # **Adversarial Robustness of AI-Generated Image D...
Embedding vector length: 1024
Embedding snippet: [0.01875837, -0.017534407, -0.0021773903, -0.012823156, 0.037036233]...
Token count: 385
----------------------------------------
Chunk index: 1
Source: 2410.01574v4.pdf
Text snippet: ## **1 Introduction**

The creation of synthetic v...
Embedding vector length: 1024
Embedding snippet: [-0.021400077, 0.0034286012, -0.0019403495, 0.025034532, 0.028112482]...
Token count: 312
----------------------------------------
Chunk index: 2
Source: 2410.01574v4.pdf
Text snippet: Since AI-generated images (AIGIs) are often widely...
Embedding vector length: 1024
Embedding snippet: [0.0134520065, -0.035696507, -0.00042847308, 0.007602457, 0.008198906]...
Token count: 361
----------------------------------------
Chunk index: 3
Source: 2410.01574v4.pdf
Text snippet: In this work, we address these limitations and eva...
Embedding vector length: 102

In [6]:
# Add the embedded document chunks to the ChromaDB collection
# Deterministic ids + upsert so re-running this cell doesn't create duplicate chunks
collection.upsert(
    ids=[f"{chunk.source}-{chunk.chunk_index}" for chunk in embedded_document],
    documents=[chunk.text for chunk in embedded_document],
    metadatas=[{"source": chunk.source, "chunk_index": chunk.chunk_index} for chunk in embedded_document],
    embeddings=[chunk.embedding for chunk in embedded_document],
)
print(f"Chunks in collection: {collection.count()}")

Chunks in collection: 46


**Observations**:
- The paper produces **46 chunks**: chunks 0-32 are the body, 33-39 are references, and 40-45 are the appendix.
- **Tables are split across chunks.** Table 1 spans chunks 15-19, and a single row can be cut mid-number (chunk 16 starts with `5|59.1|...`). A chunk containing only table rows has no caption or column headers, so an embedding has little to match against.
- The appendix tables are converted with `_._` as the decimal separator (`95_._9`), which adds noise to those chunks' embeddings.
- `upsert` with ids built from source and chunk index keeps the collection at 46 chunks, even when the cell is re-run.

## 2. Baseline: Raw Query

The question below needs two kinds of evidence: the text explaining the architecture effect (chunk 13, plus Figure 1 in chunk 19) and the Table 1 rows for the target UnivFD (chunk 15).

In [7]:
# Embed a query to retrieve relevant document chunks based on the question
query = "what architectural relationship appears to explain the difference in PGD transferability between the DRCT-CLIP → UnivFD attack and the DRCT-ConvB → UnivFD attack, and what evidence in the reported AUROC values supports this interpretation?"
query_embedding = embed_documents.embed_batch([query])[0]
print(f"Query embedding length: {len(query_embedding)}")

Query embedding length: 1024


In [8]:
chunk_text = {chunk.chunk_index: chunk.text for chunk in embedded_document}

def show_results(chunk_indices, distances=None, preview=300):
    for rank, idx in enumerate(chunk_indices):
        distance = f" | Distance: {distances[rank]:.4f}" if distances else ""
        print(f"#{rank + 1} Chunk {idx}{distance}")
        print(chunk_text[idx][:preview])
        print("-" * 40)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)
show_results([m["chunk_index"] for m in results["metadatas"][0]], results["distances"][0])

#1 Chunk 16 | Distance: 0.4008
5|59.1|36.1|44.9|24.7|
|||UnivFD|7.5|38.5|59.1|30.4|49.0|50.6|0.0|4.5|11.7|
||DRCT-CLIP|DRCT-ConvB|62.3|68.5|62.5|52.8|50.4|51.3|14.7|21.2|14.1|
|**PGD**||Corvi|63.6|71.7|63.4|53.0|50.6|51.3|18.8|26.7|14.1|
|||UnivFD|58.5|53.6|47.0|50.5|49.9|48.1|14.0|9.3|4.7|
||DRCT-ConvB|DRCT-CLIP|16.4|45.1|46.9|4
----------------------------------------
#2 Chunk 13 | Distance: 0.4429
**Attack Comparison.** Table 1 presents our robustness analysis averaged across all datasets, reporting AUROC alongside Accuracy and TPR@5%FPR. We evaluate the attacks and their transferability across different degradation levels, where “None” refers to attacks without degradation. “LTO” refers to “
----------------------------------------
#3 Chunk 41 | Distance: 0.4563
5_._8|95_._0|52_._0|55_._0|49_._4|
|||Corvi|97_._3|72_._8|78_._1|64_._9|70_._7|67_._8|73_._9|74_._7|95_._0|50_._3|63_._0|50_._5|
|PGD||UnivFD|98_._8|75_._6|61_._0|53_._2|69_._6|33_._3|32_._5|41_._1|88_._5|66_._6|67_._3|46_.

**Observations**:
- The raw question retrieves **2 of the 3 gold chunks**: the explanation (chunk 13, rank 2) and the UnivFD rows of Table 1 (chunk 15, rank 4). Figure 1's transferability matrix (chunk 19) is missed.
- **Table chunks crowd the ranking.** 4 of the 5 results are table fragments (16, 41, 15, 18). Chunk 16 (DRCT-CLIP/DRCT-ConvB target rows) is ranked first and chunk 41 (appendix, per-dataset) third. They match on model names and "AUROC" rather than on the reasoning the question asks for.
- Distances are close together (0.40–0.58), so the explanation and the table rows are nearly tied.

## 3. Query Augmentation

Both strategies use `gemma3:4b` with Ollama's `format=` parameter, which constrains generation to a JSON schema. Without it, the model adds explanations around the query that would be embedded along with it.

### 3.1 Single-query Rewriting

In [9]:
# Load model for query augmentation and answer generation from ollama

prompt = """You are an expert search query optimizer. Your task is to rewrite the user's input into a precise, self-contained search query for a vector database. Remove filler words, infer missing context from any previous turns, and use domain-specific terminology.
Return only the rewritten query as JSON: {{"query": "..."}}. No explanation, no markdown.

User Query: \"{user_query}\"

Rewritten Search Query:
"""
print("Prompt for query augmentation:")
print(prompt.format(user_query=query))

Prompt for query augmentation:
You are an expert search query optimizer. Your task is to rewrite the user's input into a precise, self-contained search query for a vector database. Remove filler words, infer missing context from any previous turns, and use domain-specific terminology.
Return only the rewritten query as JSON: {"query": "..."}. No explanation, no markdown.

User Query: "what architectural relationship appears to explain the difference in PGD transferability between the DRCT-CLIP → UnivFD attack and the DRCT-ConvB → UnivFD attack, and what evidence in the reported AUROC values supports this interpretation?"

Rewritten Search Query:



In [10]:
from ollama import chat
from pydantic import BaseModel, Field

LLM_OPTIONS = {"temperature": 0.3, "seed": 42}  # fixed seed so benchmark runs are reproducible

class QueryExpansion(BaseModel):
    queries: list[str] = Field(min_length=3, max_length=5)

class GeneralizedQuery(BaseModel):
    query: str

def generalize_query(user_query: str,prompt: str) -> str:
    response = chat(
        model="gemma3:4b",
        messages=[{"role": "user", "content": prompt.format(user_query=user_query)}],
        format=GeneralizedQuery.model_json_schema(),  # constrain output to {"query": ...} so no explanation is added
        options=LLM_OPTIONS,
        think=False,
    )
    return GeneralizedQuery.model_validate_json(response.message.content).query.strip().strip('"*')

In [11]:
generalized_query = generalize_query(query, prompt)
print("Generalized Query:")
print(generalized_query)

Generalized Query:
PGD transferability DRCT-CLIP vs DRCT-ConvB AUROC UnivFD architectural relationships


In [12]:
# Embed the generalized query and retrieve relevant document chunks from the ChromaDB collection
generalized_query_embedding = embed_documents.embed_batch([generalized_query])[0]
print(f"Generalized Query embedding length: {len(generalized_query_embedding)}")

Generalized Query embedding length: 1024


In [13]:
# print the relevant document chunks for verification
relevant_chunks = collection.query(
    query_embeddings=[generalized_query_embedding],
    n_results=5
)
show_results([m["chunk_index"] for m in relevant_chunks["metadatas"][0]], relevant_chunks["distances"][0])

#1 Chunk 41 | Distance: 0.4554
5_._8|95_._0|52_._0|55_._0|49_._4|
|||Corvi|97_._3|72_._8|78_._1|64_._9|70_._7|67_._8|73_._9|74_._7|95_._0|50_._3|63_._0|50_._5|
|PGD||UnivFD|98_._8|75_._6|61_._0|53_._2|69_._6|33_._3|32_._5|41_._1|88_._5|66_._6|67_._3|46_._6|
||DRCT-ConvB|DRCT-CLIP|98_._8|24_._8|50_._2|55_._2|69_._6|5_._6|28_._2|40
----------------------------------------
#2 Chunk 16 | Distance: 0.4616
5|59.1|36.1|44.9|24.7|
|||UnivFD|7.5|38.5|59.1|30.4|49.0|50.6|0.0|4.5|11.7|
||DRCT-CLIP|DRCT-ConvB|62.3|68.5|62.5|52.8|50.4|51.3|14.7|21.2|14.1|
|**PGD**||Corvi|63.6|71.7|63.4|53.0|50.6|51.3|18.8|26.7|14.1|
|||UnivFD|58.5|53.6|47.0|50.5|49.9|48.1|14.0|9.3|4.7|
||DRCT-ConvB|DRCT-CLIP|16.4|45.1|46.9|4
----------------------------------------
#3 Chunk 18 | Distance: 0.5888
1|
||UnivFD|LTO|28.6|79.4|65.3|35.6|69.1|57.6|0.8|34.8|16.3|
||DRCT-CLIP|LTO|0.0|62.1|60.9|0.5|50.1|50.5|0.0|12.5|14.4|
|**Universal**|DRCT-ConvB|LTO|0.1|12.5|41.7|2.4|24.6|43.8|0.0|0.0|2.3|
||Corvi|LTO|1.5|9.9|43.3|12.1|45

**Observations**:
- With the JSON schema, the rewrite is a clean keyword string: `PGD transferability DRCT-CLIP vs DRCT-ConvB AUROC UnivFD architectural relationships`.
- On this question it is **worse than the baseline**: only chunk 13 is retrieved, and it drops from rank 2 to rank 5. Chunk 15 is lost.
- The rewrite is mostly entity names, so the embedding is pulled further toward table chunks: 4 of the top 5 are table rows (41, 16, 18, 44), including two appendix chunks. Compressing the question removed the "explain why" part that matches the prose in chunk 13.

### 3.2 Multi-query Expansion

In [14]:
multi_query_prompt = """You are a retrieval assistant for a corpus of academic research papers (machine learning, computer vision, security).
Your job: rewrite the user's question into {n} different search queries that together maximize the chance of retrieving every passage needed to answer it from a vector database.

Each query must take a DIFFERENT angle:
1. DECOMPOSITION - if the question asks several things, split it: one query per sub-question.
2. TERMINOLOGY - rephrase using the technical vocabulary, metric names, and section language a paper would actually use (e.g. "results table", "ablation", "transferability", "backbone").
3. STEP-BACK - a broader query about the underlying concept, method, or setup the question depends on.
4. ANSWER-STYLE - a declarative sentence phrased the way the answer would appear in the paper's text (not a question). Do not invent numbers or facts; use placeholders like "X" if needed.

Rules:
- Each query must be self-contained and understandable without the original question.
- Copy model names, dataset names, acronyms, and metrics EXACTLY as written (e.g. "DRCT-CLIP", "UnivFD", "AUROC"). Never expand or alter them unless you are certain.
- Do not add facts, entities, or assumptions that are not in the original question.
- Keep each query under 30 words. No near-duplicates.
- Return between 3 and 5 queries.

Question: "{user_query}"
Output:
"""

In [15]:
def expand_query(user_query: str,prompt: str, n: int = 4) -> list[str]:
    response = chat(
        model="gemma3:4b",
        messages=[{"role": "user", "content": prompt.format(user_query=user_query, n=n)}],
        format=QueryExpansion.model_json_schema(),
        options=LLM_OPTIONS,
        think=False,
    )
    queries = QueryExpansion.model_validate_json(response.message.content).queries
    return [user_query] + queries  # always keep the original query

In [16]:
expanded = expand_query(query,multi_query_prompt)
for q in expanded:
    print("-", q)

- what architectural relationship appears to explain the difference in PGD transferability between the DRCT-CLIP → UnivFD attack and the DRCT-ConvB → UnivFD attack, and what evidence in the reported AUROC values supports this interpretation?
- architectural relationships between DRCT-CLIP and DRCT-ConvB concerning UnivFD transferability.
- Evaluate the impact of backbones (DRCT-CLIP, DRCT-ConvB) on PGD attack transferability using AUROC.
- Investigate the transferability of adversarial examples generated via PGD attacks.
- Report AUROC values for DRCT-CLIP and DRCT-ConvB attacks to assess transferability differences.


Each query retrieves its own ranked list. **Reciprocal Rank Fusion** merges them by giving every chunk a score of `1 / (60 + rank)` per list it appears in. Chunks ranked high by several queries come first, and no distance calibration is needed across queries.

In [17]:
def retrieve_ids(query_embeddings, n_results):
    """Return one ranked list of chunk indices per query embedding."""
    res = collection.query(query_embeddings=query_embeddings, n_results=n_results, include=["metadatas"])
    return [[m["chunk_index"] for m in metas] for metas in res["metadatas"]]

def rrf_fuse(ranked_lists, k, rrf_k=60):
    scores = {}
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked):
            scores[idx] = scores.get(idx, 0) + 1 / (rrf_k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)[:k]

def multi_query_retrieve(queries, k=5, per_query=10):
    return rrf_fuse(retrieve_ids(embed_documents.embed_batch(queries), per_query), k)

show_results(multi_query_retrieve(expanded))

#1 Chunk 13
**Attack Comparison.** Table 1 presents our robustness analysis averaged across all datasets, reporting AUROC alongside Accuracy and TPR@5%FPR. We evaluate the attacks and their transferability across different degradation levels, where “None” refers to attacks without degradation. “LTO” refers to “
----------------------------------------
#2 Chunk 41
5_._8|95_._0|52_._0|55_._0|49_._4|
|||Corvi|97_._3|72_._8|78_._1|64_._9|70_._7|67_._8|73_._9|74_._7|95_._0|50_._3|63_._0|50_._5|
|PGD||UnivFD|98_._8|75_._6|61_._0|53_._2|69_._6|33_._3|32_._5|41_._1|88_._5|66_._6|67_._3|46_._6|
||DRCT-ConvB|DRCT-CLIP|98_._8|24_._8|50_._2|55_._2|69_._6|5_._6|28_._2|40
----------------------------------------
#3 Chunk 40
|||||**Syn**|**thbuster**|||**Ch**|**ameleon**|||**G**|**PT-4o**||
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|**Attack**|**Target**|**Source**|**Clean**||**Attacked**||**Clean**||**Attacked**||**Clean**||**Attacked**||
|||||**No Deg.**|**Med Deg.**|**High Deg.

**Observations**:
- The expansion does split the question (a separate query for DRCT-CLIP → UnivFD and for DRCT-ConvB → UnivFD) and adds a step-back query about backbone architecture.
- After fusion, chunk 13 moves up to **rank 1**, because every query retrieves it. But chunk 15 drops out, and the other 4 slots are all table fragments (16, 41, 40, 44). Recall is 1/3, the same as the rewrite and below the baseline.
- **RRF rewards chunks that appear in many lists.** Table chunks mention every model name and "AUROC", so they rank moderately well for *every* query, and fusion pushes them up. This "hub" effect is the main weakness of multi-query on this corpus (confirmed in Section 4).

## 4. Evaluation: Baseline vs. Rewriting vs. Multi-query

**Protocol:**
- The corpus, chunks, embeddings and collection from Section 1 are reused unchanged. Only the query side differs between strategies.
- **Test set:** 18 questions about the paper, written by reading the 46 chunks. Each one is labeled with the chunk indices needed to answer it (`gold`). They mix single-passage lookups, multi-part questions spanning 2–3 chunks, and questions answered by table or appendix chunks.
- Each strategy returns 5 chunk indices per question, which are scored against `gold`:
  - **Hit@1 / Hit@5:** whether a gold chunk is ranked first / anywhere in the top 5.
  - **Recall@5:** the share of gold chunks in the top 5. This is the metric that matters for multi-part questions.
  - **MRR:** 1 / rank of the first gold chunk (0 if none is retrieved).
- **Latency:** wall-clock time for the query side (LLM call + embedding + search).
- The LLM is run with `seed=42`, so re-running gives the same rewrites.

**Caveat:** 18 questions on one paper is a small sample, where one question is worth ~5.6 points of Hit@5. Treat differences of 1–2 questions as noise.

In [18]:
TEST_SET = [
    # (question, gold chunk indices, question type)
    ("What architectural relationship appears to explain the difference in PGD transferability between the DRCT-CLIP → UnivFD attack and the DRCT-ConvB → UnivFD attack, and what evidence in the reported AUROC values supports this interpretation?", {13, 15, 19}, "multi-part"),
    ("Which three datasets are used to evaluate the detectors, and where do the real images for the GPT-4o dataset come from?", {7}, "lookup"),
    ("What backbone does the Corvi detector use, and what architectural change helps it preserve high-frequency information?", {8}, "lookup"),
    ("How does diffusion reconstruction contrastive training (DRCT) work and which backbones does it support?", {8}, "lookup"),
    ("Which five attack algorithms are evaluated, and which of them is query-based rather than transfer-based?", {9}, "lookup"),
    ("What image transformations are applied in the medium and high degradation settings?", {10, 14}, "multi-part"),
    ("Why do the authors use AUROC as their primary metric and exclude attack success rate?", {11}, "lookup"),
    ("What perturbation budget, step size and number of iterations are used for the attacks, and are 10 PGD steps enough to converge?", {12, 45}, "multi-part"),
    ("How effective is the Leave-Target-Out ensemble attack compared to PGD, and which detector remains most resilient to it?", {13, 17}, "multi-part"),
    ("How does the universal perturbation attack compare to PGD and the query-efficient attack in average AUROC?", {20}, "lookup"),
    ("Why are CNN-based detectors more affected by social media degradations than CLIP-based detectors?", {22}, "lookup"),
    ("Why is the Diverse Input attack more robust to image degradation than the Ensemble attack?", {22}, "lookup"),
    ("What is the trade-off between adversarial image quality and attack effectiveness, and which attack has the best quality score?", {23, 24, 25}, "multi-part"),
    ("How much did the HIVE commercial detector's AUROC drop under attack on Synthbuster and Chameleon?", {25, 26}, "multi-part"),
    ("How does replacing UnivFD's CLIP backbone with RobustCLIP affect adversarial robustness and clean AUROC?", {27, 30}, "multi-part"),
    ("What limitations of their study do the authors acknowledge?", {31}, "lookup"),
    ("What assumptions does the threat model make about the attacker's knowledge of the detector?", {6}, "lookup"),
    ("On the Chameleon dataset, what AUROC does UnivFD reach under a PGD attack crafted on DRCT-CLIP?", {40}, "table"),
]
print(f"{len(TEST_SET)} questions | " + ", ".join(
    f"{t}: {sum(1 for *_, qt in TEST_SET if qt == t)}" for t in ("lookup", "multi-part", "table")
))

18 questions | lookup: 10, multi-part: 7, table: 1


In [ ]:
import time

K = 5            # chunks returned to the generator
PER_QUERY = 10   # candidates per query before RRF fusion in multi-query

def baseline_strategy(question):
    return retrieve_ids(embed_documents.embed_batch([question]), K)[0], [question]

def rewrite_strategy(question):
    rewritten = generalize_query(question, prompt)
    return retrieve_ids(embed_documents.embed_batch([rewritten]), K)[0], [rewritten]

def multi_query_strategy(question):
    queries = expand_query(question, multi_query_prompt)
    return multi_query_retrieve(queries, k=K, per_query=PER_QUERY), queries

STRATEGIES = {
    "baseline": baseline_strategy,
    "rewrite": rewrite_strategy,
    "multi-query": multi_query_strategy,
}

def score(retrieved, gold):
    first_hit = next((rank for rank, idx in enumerate(retrieved) if idx in gold), None)
    return {
        "hit@1": float(retrieved[0] in gold),                       # Hit rate at rank 1
        "hit@5": float(first_hit is not None),                      # Hit rate at rank 5
        "recall@5": len(gold & set(retrieved)) / len(gold),         # Recall at rank 5
        "mrr": 0.0 if first_hit is None else 1 / (first_hit + 1),   # Mean Reciprocal Rank
    }

In [20]:
import pandas as pd

rows = []
for q_id, (question, gold, q_type) in enumerate(TEST_SET):
    for name, strategy in STRATEGIES.items():
        start = time.perf_counter()
        retrieved, queries = strategy(question)
        latency = time.perf_counter() - start
        rows.append({
            "q_id": q_id, "type": q_type, "strategy": name,
            "gold": sorted(gold), "retrieved": retrieved, "queries": queries,
            "latency_s": latency, **score(retrieved, gold),
        })

eval_df = pd.DataFrame(rows)
print(f"Evaluated {eval_df['q_id'].nunique()} questions x {eval_df['strategy'].nunique()} strategies")

Evaluated 18 questions x 3 strategies


### 4.1 Overall Results

In [21]:
metrics = ["hit@1", "hit@5", "recall@5", "mrr", "latency_s"]
order = list(STRATEGIES)
summary = eval_df.groupby("strategy")[metrics].mean().loc[order]
summary.round(3)

,hit@1,hit@5,recall@5,mrr,latency_s
strategy,,,,,
baseline,0.556,0.778,0.731,0.657,0.257
rewrite,0.611,0.889,0.769,0.719,5.507
multi-query,0.333,0.778,0.731,0.488,24.271


In [22]:
# Recall@5 by question type
eval_df.pivot_table(index="type", columns="strategy", values="recall@5", aggfunc="mean")[order].round(3)

strategy,baseline,rewrite,multi-query
type,,,
lookup,0.700,0.80,0.800
multi-part,0.881,0.69,0.738
table,0.000,1.00,0.000


### 4.2 Per-question Breakdown

In [23]:
per_question = eval_df.pivot_table(index="q_id", columns="strategy", values="recall@5")[order]
per_question.insert(0, "gold", [sorted(g) for _, g, _ in TEST_SET])
per_question.insert(0, "question", [q[:70] for q, _, _ in TEST_SET])
per_question.round(2)

strategy,question,gold,baseline,rewrite,multi-query
q_id,,,,,
0,What architectural relationship appears to exp...,"[13, 15, 19]",0.67,0.33,1.00
1,Which three datasets are used to evaluate the ...,[7],1.00,1.00,1.00
2,"What backbone does the Corvi detector use, and...",[8],1.00,1.00,1.00
3,How does diffusion reconstruction contrastive ...,[8],0.00,1.00,1.00
4,"Which five attack algorithms are evaluated, an...",[9],1.00,1.00,1.00
5,What image transformations are applied in the ...,"[10, 14]",1.00,0.50,0.00
6,Why do the authors use AUROC as their primary ...,[11],1.00,1.00,1.00
7,"What perturbation budget, step size and number...","[12, 45]",1.00,1.00,1.00
8,How effective is the Leave-Target-Out ensemble...,"[13, 17]",0.50,0.50,0.50


In [24]:
# Questions where the strategies disagree: compare the chunks each retrieved and the queries the LLM generated
for q_id in per_question.index[per_question[order].nunique(axis=1) > 1]:
    question, gold, _ = TEST_SET[q_id]
    print(f"Q{q_id}: {question}\n  gold: {sorted(gold)}")
    for _, row in eval_df[eval_df["q_id"] == q_id].iterrows():
        print(f"  {row['strategy']:<12} recall@5={row['recall@5']:.2f} retrieved={row['retrieved']}")
        if row["strategy"] != "baseline":
            for q in row["queries"][1 if row["strategy"] == "multi-query" else 0:]:
                print(f"      - {q}")
    print()

Q0: What architectural relationship appears to explain the difference in PGD transferability between the DRCT-CLIP → UnivFD attack and the DRCT-ConvB → UnivFD attack, and what evidence in the reported AUROC values supports this interpretation?
  gold: [13, 15, 19]
  baseline     recall@5=0.67 retrieved=[16, 13, 41, 15, 40]
  rewrite      recall@5=0.33 retrieved=[41, 16, 18, 44, 13]
      - PGD transferability DRCT-CLIP vs. DRCT-ConvB AUROC UnivFD architectural relationships
  multi-query  recall@5=1.00 retrieved=[13, 40, 15, 19, 18]
      - Architectural relationships between DRCT-CLIP and DRCT-ConvB regarding transferability.
      - Evaluate the impact of UnivFD as a metric for assessing PGD attack transferability.
      - Investigate the role of backbones (DRCT-CLIP, DRCT-ConvB) in adversarial robustness.
      - AUROC values demonstrate a correlation between attack architecture and transferability performance.

Q3: How does diffusion reconstruction contrastive training (DRCT) work 

**Observations**:

**Overall** (18 questions, k = 5):

| Strategy | Hit@1 | Hit@5 | Recall@5 | MRR | Latency |
|---|---|---|---|---|---|
| Baseline | 0.556 | 0.778 | 0.731 | 0.657 | 0.2 s |
| Rewrite | **0.611** | **0.889** | **0.769** | **0.719** | 4.8 s |
| Multi-query | 0.389 | 0.667 | 0.528 | 0.485 | 18.4 s |

- **Rewriting is slightly ahead of the baseline, but the gap is within noise.** It wins 2 questions and loses 3 on Recall@5. Its net gain in Hit@5 is 2 questions (0.778 → 0.889).
- **Multi-query is clearly the worst**, and it costs ~80× the baseline latency. It beats the baseline only on Q17, a table lookup. It loses all of the gold chunks on 3 questions where the baseline found them at rank 1–2 (Q2, Q4, Q9), and loses part of the gold set on Q0, Q5, Q12 and Q14.

**Where rewriting helps: acronyms and table lookups.**
- Q3 (*"How does DRCT work…"*): the raw question contains "DRCT" and gets pulled toward table rows labeled DRCT-CLIP/DRCT-ConvB, so the baseline misses chunk 8. The rewrite `diffusion reconstruction contrastive training backbone support` drops the acronym and finds chunk 8 at rank 1.
- Q17 (Chameleon/UnivFD/PGD AUROC): the keyword rewrite matches the appendix table header (chunk 40), which the natural-language question misses.

**Where rewriting hurts: multi-part questions.** Recall@5 on multi-part questions falls from 0.881 to 0.690. A single compressed query keeps one part of the question and drops the other: Q5 loses the Table 1 caption (chunk 14), Q14 loses the defense results (chunk 30), and Q0 loses Table 1 (chunk 15).

**Why multi-query underperforms**, from the generated queries in the cell above:
1. **The original query is diluted.** It counts for 1 of 5 ranked lists in RRF. A strong rank-1 hit from the original (e.g. chunk 8 for Q2, chunk 9 for Q4) is outvoted by chunks that appear moderately high in many lists.
2. **Hub chunks win the fusion.** Table fragments (16, 18, 40, 41, 44, 45) contain every model name and metric, so they show up in every query's top 10 and RRF ranks them highest.
3. **The step-back and answer-style queries drift off the paper.** Examples: *"Evaluating the effectiveness of adversarial attacks on deep learning models"* (Q9), *"Comparing… attacks like PGD and CW"* (Q12, where CW is not in the paper), *"Impact of transformations on the performance of the DRCT-CLIP model"* (Q5), and the unfilled placeholder *"[architectural change]"* (Q2). A 4B model can't keep generic angles anchored to the source.

**Failures for every strategy:** Q10 (CNN vs. CLIP degradation sensitivity, chunk 22) and Q15 (limitations, chunk 31). In both, the answer sits mid-chunk under an unrelated heading: chunk 22 starts with the Figure 3 caption, and chunk 31 starts with "Detection vs. Generation Arms Race". Query rewording can't fix that; better chunk boundaries or a reranker might.